---

## 🔧 Step 1: Environment Setup

In [ ]:
%%time
import os
import sys
import shutil
from pathlib import Path
import subprocess

# Configuration
REPO_INPUT = Path('/kaggle/input/gdsearch-repository')
WORKING_DIR = Path('/kaggle/working/GDSearch')
OUTPUT_DIR = Path('/kaggle/working/results')

print("="*80)
print("🚀 GDSearch Kaggle Environment Setup")
print("="*80)

# Check if repository exists
if not REPO_INPUT.exists():
    print("❌ ERROR: Repository not found at /kaggle/input/gdsearch-repository")
    print("\n📋 Instructions:")
    print("   1. Upload GDSearch repository as a Kaggle dataset")
    print("   2. Add dataset to this notebook")
    print("   3. Ensure it's mounted at /kaggle/input/gdsearch-repository")
    raise FileNotFoundError("GDSearch repository not found")

print(f"✅ Repository found: {REPO_INPUT}")
print(f"📁 Working directory: {WORKING_DIR}")
print(f"💾 Output directory: {OUTPUT_DIR}")

### Copy Repository to Working Directory

In [ ]:
%%time
print("📦 Copying repository to working directory...")

# Remove existing working directory if present
if WORKING_DIR.exists():
    print(f"⚠️  Removing existing {WORKING_DIR}")
    shutil.rmtree(WORKING_DIR)

# Copy repository
shutil.copytree(REPO_INPUT, WORKING_DIR, symlinks=False, ignore=None)
print(f"✅ Repository copied to {WORKING_DIR}")

# Change to working directory
os.chdir(WORKING_DIR)
print(f"📂 Current directory: {os.getcwd()}")

# Add to Python path
if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))
    print(f"✅ Added {WORKING_DIR} to Python path")

# Verify key files
key_files = ['run_all_kaggle.py', 'requirements.txt', 'src/__init__.py']
for file in key_files:
    if (WORKING_DIR / file).exists():
        print(f"✅ Found {file}")
    else:
        print(f"❌ Missing {file}")

### Install Dependencies

In [ ]:
%%time
print("📦 Installing dependencies...")
print("="*80)

# Check if requirements_kaggle.txt exists, otherwise use requirements.txt
if (WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt').exists():
    requirements_file = WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt'
    print(f"✅ Using Kaggle-specific requirements: {requirements_file}")
elif (WORKING_DIR / 'requirements.txt').exists():
    requirements_file = WORKING_DIR / 'requirements.txt'
    print(f"✅ Using standard requirements: {requirements_file}")
else:
    raise FileNotFoundError("No requirements file found")

# Install dependencies (suppress output for cleaner notebook)
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✅ Dependencies installed successfully")
else:
    print("⚠️  Some dependencies may have failed:")
    print(result.stderr)

print("="*80)

### Verify Environment

In [ ]:
print("🔍 Verifying environment...")
print("="*80)

# Check Python version
print(f"🐍 Python: {sys.version.split()[0]}")

# Check PyTorch and CUDA
import torch
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - CUDA version: {torch.version.cuda}")
    print(f"   - GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"     Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

# Check key dependencies
try:
    import numpy as np
    print(f"✅ NumPy: {np.__version__}")
except ImportError as e:
    print(f"❌ NumPy: {e}")

try:
    import pandas as pd
    print(f"✅ Pandas: {pd.__version__}")
except ImportError as e:
    print(f"❌ Pandas: {e}")

try:
    import matplotlib
    print(f"✅ Matplotlib: {matplotlib.__version__}")
except ImportError as e:
    print(f"❌ Matplotlib: {e}")

try:
    import tqdm
    print(f"✅ tqdm: {tqdm.__version__}")
except ImportError as e:
    print(f"❌ tqdm: {e}")

try:
    import mlflow
    print(f"✅ MLflow: {mlflow.__version__}")
except ImportError as e:
    print(f"❌ MLflow: {e}")

# Verify GDSearch modules
try:
    from src.core import optimizers
    print(f"✅ GDSearch core modules imported")
except ImportError as e:
    print(f"❌ GDSearch import error: {e}")

print("="*80)
print("✅ Environment setup complete!")

---

## 🧪 Step 2: Quick Validation Test

In [ ]:
%%time
print("🧪 Running quick validation test...")
print("="*80)

# Run quick validation (import-safe, fast)
result = subprocess.run(
    [sys.executable, 'scripts/quick_validation_test.py', '--verbose'],
    capture_output=True,
    text=True,
    cwd=WORKING_DIR
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️  Validation warnings:")
    print(result.stderr)
else:
    print("\n✅ Quick validation passed!")

print("="*80)

---

## 🚀 Step 3: Run Experiments

### Configuration Options

Choose your experiment mode by uncommenting the desired configuration below.

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

# ===== Option 1: Quick Test (5 minutes) =====
# Fast smoke test - 2 epochs, 3 seeds, MNIST only
EXPERIMENT_MODE = 'quick'
EXPERIMENTS = 'mnist'
SEEDS = '42,123,456'
EXTRA_ARGS = ['--ultra-quick']

# ===== Option 2: Proposal-Required Experiments (2-3 hours) =====
# All experiments needed for research proposal
# EXPERIMENT_MODE = 'proposal'
# EXPERIMENTS = 'mnist,2d,hyperparam_sensitivity,convergence_validation,theory_practice'
# SEEDS = '42,123,456,789,1011'
# EXTRA_ARGS = ['--kaggle-t4', '--time-budget', '3.0']

# ===== Option 3: Full Benchmark Suite (8-10 hours) =====
# All experiments with 10 seeds for statistical validation
# EXPERIMENT_MODE = 'full'
# EXPERIMENTS = 'all'
# SEEDS = '42,123,456,789,1011,1213,1415,1617,1819,2021'
# EXTRA_ARGS = ['--kaggle-t4', '--time-budget', '10.0']

# ===== Option 4: Custom Configuration =====
# Customize experiments, seeds, and arguments
# EXPERIMENT_MODE = 'custom'
# EXPERIMENTS = 'mnist,cifar10,2d'  # Choose: mnist, cifar10, 2d, nlp, medical, etc.
# SEEDS = '42,123,456'  # Minimum 3 seeds for statistics
# EXTRA_ARGS = ['--kaggle-t4', '--time-budget', '5.0']

# =============================================================================
# Results Directory
# =============================================================================
RESULTS_DIR = OUTPUT_DIR / f'results_{EXPERIMENT_MODE}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📊 Experiment Mode: {EXPERIMENT_MODE.upper()}")
print(f"🧪 Experiments: {EXPERIMENTS}")
print(f"🎲 Seeds: {SEEDS}")
print(f"⚙️  Extra Args: {' '.join(EXTRA_ARGS)}")
print(f"💾 Results will be saved to: {RESULTS_DIR}")

### Execute Experiments

In [ ]:
%%time
print("="*80)
print(f"🚀 Starting {EXPERIMENT_MODE.upper()} mode experiments")
print("="*80)

# Build command
cmd = [
    sys.executable,
    'run_all_kaggle.py',
    '--experiments', EXPERIMENTS,
    '--seeds', SEEDS,
    '--results-dir', str(RESULTS_DIR)
] + EXTRA_ARGS

print("📝 Command:")
print(' '.join(cmd))
print("\n" + "="*80)

# Run experiments
import time
start_time = time.time()

try:
    result = subprocess.run(
        cmd,
        cwd=WORKING_DIR,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True
    )
    
    # Print output
    print(result.stdout)
    
    elapsed = time.time() - start_time
    print("\n" + "="*80)
    print(f"✅ Experiments completed successfully!")
    print(f"⏱️  Total time: {elapsed/3600:.2f} hours ({elapsed/60:.1f} minutes)")
    print("="*80)
    
except subprocess.CalledProcessError as e:
    elapsed = time.time() - start_time
    print(f"\n❌ Experiments failed after {elapsed/60:.1f} minutes")
    print("Error output:")
    print(e.stdout)
    raise

---

## 📊 Step 4: Results Analysis

### List Generated Results

In [ ]:
import os
from pathlib import Path

print("📁 Generated Results:")
print("="*80)

# List all result directories
result_dirs = [
    'experiments',
    '2d_optimization',
    'beta_sensitivity',
    'hyperparameter_sensitivity',
    'theory_practice',
    'visualizations',
    'analysis',
    'reports'
]

for dir_name in result_dirs:
    dir_path = RESULTS_DIR / dir_name
    if dir_path.exists():
        file_count = sum(1 for _ in dir_path.rglob('*') if _.is_file())
        print(f"✅ {dir_name}: {file_count} files")
        
        # Show first few files
        files = sorted(dir_path.rglob('*.csv'))[:5]
        if files:
            for f in files:
                rel_path = f.relative_to(RESULTS_DIR)
                print(f"   - {rel_path}")
            if len(list(dir_path.rglob('*.csv'))) > 5:
                print(f"   ... and {len(list(dir_path.rglob('*.csv'))) - 5} more CSV files")
    else:
        print(f"⚠️  {dir_name}: not found")

print("="*80)

### Quick Results Preview

In [ ]:
import pandas as pd
import glob

print("📈 Quick Results Preview:")
print("="*80)

# Find MNIST results
mnist_csvs = list((RESULTS_DIR / 'experiments' / 'mnist').glob('*.csv'))

if mnist_csvs:
    print(f"\n📊 Found {len(mnist_csvs)} MNIST result files\n")
    
    # Load and display summary
    results = []
    for csv in mnist_csvs[:10]:  # Show first 10
        df = pd.read_csv(csv)
        if len(df) > 0:
            final_row = df.iloc[-1]
            results.append({
                'file': csv.name,
                'epochs': len(df),
                'final_train_loss': final_row.get('train_loss', 'N/A'),
                'final_test_acc': final_row.get('test_acc', 'N/A'),
                'final_grad_norm': final_row.get('grad_norm', 'N/A')
            })
    
    if results:
        summary_df = pd.DataFrame(results)
        print(summary_df.to_string(index=False))
        
        # Check for grad_norm column
        if 'final_grad_norm' in summary_df.columns:
            has_grad_norm = summary_df['final_grad_norm'] != 'N/A'
            if has_grad_norm.all():
                print("\n✅ All results include gradient norm tracking!")
            else:
                print("\n⚠️  Some results missing gradient norm")
else:
    print("⚠️  No MNIST results found")

print("\n" + "="*80)

### Display Visualizations

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

print("🎨 Visualizations:")
print("="*80)

# Find visualization PNGs
viz_dir = RESULTS_DIR / 'visualizations' / 'static'
if viz_dir.exists():
    png_files = sorted(viz_dir.rglob('*.png'))[:5]  # Show first 5
    
    if png_files:
        for png in png_files:
            print(f"\n📊 {png.name}")
            try:
                display(Image(filename=str(png), width=800))
            except Exception as e:
                print(f"   ⚠️  Could not display: {e}")
    else:
        print("⚠️  No visualization PNGs found")
else:
    print("⚠️  Visualization directory not found")

print("\n" + "="*80)

---

## 💾 Step 5: Save Results for Download

In [ ]:
%%time
print("💾 Preparing results for download...")
print("="*80)

# Create archive
import shutil
archive_name = f'gdsearch_results_{EXPERIMENT_MODE}'
archive_path = OUTPUT_DIR / archive_name

print(f"📦 Creating archive: {archive_name}.zip")
shutil.make_archive(str(archive_path), 'zip', RESULTS_DIR)

# Get archive size
archive_file = f"{archive_path}.zip"
size_mb = os.path.getsize(archive_file) / (1024 * 1024)
print(f"✅ Archive created: {size_mb:.2f} MB")

# Summary
print("\n" + "="*80)
print("📁 Results saved to:")
print(f"   - Directory: {RESULTS_DIR}")
print(f"   - Archive: {archive_file}")
print("\n📥 To download:")
print("   1. Check the 'Output' tab in Kaggle")
print(f"   2. Download {archive_name}.zip")
print("   3. Extract and analyze locally")
print("="*80)

---

## 📋 Step 6: Experiment Summary Report

In [ ]:
# Check for auto-generated summary report
summary_report = RESULTS_DIR / 'reports' / '00_EXPERIMENT_SUMMARY.md'

print("📋 Experiment Summary:")
print("="*80)

if summary_report.exists():
    with open(summary_report, 'r') as f:
        print(f.read())
else:
    print("⚠️  Summary report not found")
    print("\nManual Summary:")
    print(f"- Mode: {EXPERIMENT_MODE}")
    print(f"- Experiments: {EXPERIMENTS}")
    print(f"- Seeds: {SEEDS}")
    print(f"- Results directory: {RESULTS_DIR}")

print("\n" + "="*80)

---

## ✅ Completion Checklist

After running this notebook, verify:

- [ ] Environment setup completed without errors
- [ ] Quick validation test passed
- [ ] Experiments ran successfully
- [ ] Results generated in expected directories
- [ ] CSV files contain required columns (grad_norm, test_acc, etc.)
- [ ] Visualizations generated (if applicable)
- [ ] Results archive created for download

---

## 🔧 Troubleshooting

### Common Issues:

**1. Repository not found**
```python
# Check dataset mounting:
!ls /kaggle/input/
```

**2. Out of Memory (OOM)**
```python
# Reduce batch size or use ultra-quick mode
EXTRA_ARGS = ['--ultra-quick', '--batch-size', '32']
```

**3. Time limit exceeded**
```python
# Reduce time budget or number of seeds
SEEDS = '42,123,456'  # Use fewer seeds
EXTRA_ARGS = ['--time-budget', '3.0']  # Lower budget
```

**4. Missing dependencies**
```python
# Manually install missing package
!pip install <package-name>
```

---

## 📚 Additional Resources

- **Documentation:** See `README.md` in repository
- **Proposal Compliance:** See `docs/PROPOSAL_COMPLIANCE_CHECKLIST.md`
- **Configuration Schema:** See `configs/config_schema.json`

---

**Generated by GDSearch Kaggle Runner**  
*Last Updated: December 23, 2025*